In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_openai import ChatOpenAI
import operator
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.checkpoint.memory import MemorySaver
import time

llm = ChatOpenAI(
    model="gpt-4.1-mini",  # Your Azure deployment name
    base_url="https://openai-rg-nadeem.openai.azure.com/openai/v1",
    #add your api key below
    )

class State(TypedDict):
    human: str
    ai: str
    history: Annotated[list[BaseMessage], operator.add]

def chat_node(state: State):
    prompt = f"you have all the history as {state['history']} and you have to answer the question: {state['human']}"
    result = llm.invoke(prompt)
    answer = result.content
    print("AI:", answer)
    time.sleep(4)
    
    return {
        "ai":answer,
        "history": [
        HumanMessage(content=state["human"]),
        AIMessage(content=answer)
    ] }

def print_history(state: State):
    print(state['history'])
    return {}
     
graph = StateGraph(State)
graph.add_node('chat_node', chat_node)
graph.add_node('print_history', print_history)
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', 'print_history')
graph.add_edge('print_history', END)
memory = MemorySaver()

workflow = graph.compile(
    checkpointer= memory
)

config = {
    "configurable": {
        "thread_id": "nadeem"
    }
}

while True:
    question = str(input('YOU:'))
    if question =="end":
        initial_state = {
        "human": question,
        "history":[]
            }
        workflow.invoke(initial_state,
                    config=config)
        break
    initial_state = {
        "human": question,
        "history":[]
    }
    workflow.invoke(initial_state,
                    config=config)
        

this is almost same code fault tolerance

In [ ]:
# workflow.get_state({"configurable": {"thread_id": "nadeem"}})
# history = list(workflow.get_state_history({"configurable":{"thread_id": "nadeem"}}))
# print(history)
# workflow.invoke(None, config =  {"configurable": {"thread_id": "nadeem"}})

StateSnapshot(values={'human': 'kaha se ho', 'ai': 'Main ek AI hoon, isliye mera koi physical sthaan nahi hai. Lekin main yahan aapki madad ke liye hoon! Aap kaha se hain?', 'history': [HumanMessage(content='kia nam hai tumhara', additional_kwargs={}, response_metadata={}), AIMessage(content='Mera naam ChatGPT hai. Aap se milkar khushi hui! Aapka naam kya hai?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='kaha se ho', additional_kwargs={}, response_metadata={}), AIMessage(content='Main ek AI hoon, isliye mera koi physical sthaan nahi hai. Lekin main yahan aapki madad ke liye hoon! Aap kaha se hain?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}, next=(), config={'configurable': {'thread_id': 'nadeem', 'checkpoint_id': '1f17f547-e97a-652d-8004-5b631957a489'}}, metadata={'source': 'loop', 'step': 4, 'parents': {}}, created_at='2026-07-14T07:20:30.304360+00:00', parent_config={'configurable': {'thread_id': 'nadeem', 'checkpoint_ns': '', 'checkpoint_id': '1f17f547-ae2a-64aa-8003-7410702c1f82'}}, tasks=(), interrupts=())

This is not having the checkpoint_id which is used to time travel... time travelling is actually to run from any specific node

In [ ]:
workflow.invoke(None, config =  {"configurable": {"thread_id": "nadeem", 'checkpoint_id': '1f17f547-e97a-652d-8004-5b631957a489'}})

{'human': 'kaha se ho',
 'ai': 'Main ek AI hoon, isliye mera koi physical sthaan nahi hai. Lekin main yahan aapki madad ke liye hoon! Aap kaha se hain?',
 'history': [HumanMessage(content='kia nam hai tumhara', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Mera naam ChatGPT hai. Aap se milkar khushi hui! Aapka naam kya hai?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='kaha se ho', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Main ek AI hoon, isliye mera koi physical sthaan nahi hai. Lekin main yahan aapki madad ke liye hoon! Aap kaha se hain?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}

this is having the check point id which will be used to go to a specific node

In [ ]:
workflow.update_state({"configurable":{
    "thread_id":"nadeem",
    'checkpoint_id': '1f17f547-e97a-652d-8004-5b631957a489',
    'checkpoint_ns': ''
}},{"human":"tumhay pta mei kon hu"})

{'configurable': {'thread_id': 'nadeem',
  'checkpoint_ns': '',
  'checkpoint_id': '1f17f571-a4a9-6893-8005-cf6b2bdff77c'}}

In [ ]:
workflow.invoke(None, config =  {"configurable": {"thread_id": "nadeem", 'checkpoint_id': '1f17f571-a4a9-6893-8005-cf6b2bdff77c'}})

{'human': 'tumhay pta mei kon hu',
 'ai': 'Main ek AI hoon, isliye mera koi physical sthaan nahi hai. Lekin main yahan aapki madad ke liye hoon! Aap kaha se hain?',
 'history': [HumanMessage(content='kia nam hai tumhara', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Mera naam ChatGPT hai. Aap se milkar khushi hui! Aapka naam kya hai?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='kaha se ho', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Main ek AI hoon, isliye mera koi physical sthaan nahi hai. Lekin main yahan aapki madad ke liye hoon! Aap kaha se hain?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}